# CodeGen-LLM — Full 5-Sample Regeneration (corrected protocol)

Produces the **publication-grade** pass@1 / pass@5 numbers that replace the single-sample proxy.
Generates all 5 completions/task for both models under their correct generation protocol
(CodeGen = raw prompt, Qwen = chat template), then scores **naive vs. normalized** and reports the delta.

**Before you run:**
1. `Runtime ▸ Change runtime type ▸ GPU` (T4 is fine).
2. Run cells top to bottom. The generation cell checkpoints to Google Drive every 5 tasks,
   so if Colab disconnects, just re-run that cell — it resumes where it stopped.
3. Keep the tab open. Expected total runtime ≈ 1.5–3 h on a T4.


### Step 0 — confirm GPU

In [1]:
import torch, subprocess
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > GPU"
print("GPU ready:", torch.cuda.get_device_name(0))

Tesla T4, 15360 MiB
GPU ready: Tesla T4


### Step 1 — clone your repo (brings the LoRA adapter, HumanEval tasks, and scripts)

In [2]:
import os
if not os.path.exists("/content/CodeGen-LLM"):
    !git clone --depth 1 https://github.com/Nishviprp/CodeGen-LLM.git /content/CodeGen-LLM
%cd /content/CodeGen-LLM
print("adapter present:", os.path.exists("finetuned-adapter/adapter_config.json"))

Cloning into '/content/CodeGen-LLM'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 60 (delta 10), reused 53 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 14.64 MiB | 10.98 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/CodeGen-LLM
adapter present: True


### Step 2 — install dependencies

In [3]:
!pip -q install -U "transformers>=4.45" "peft>=0.13" accelerate datasets
print("deps installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.6 MB/s eta 0:00:00
deps installed


In [7]:
print('Installing torchao to resolve dependency error')
!pip install --upgrade torchao
print('torchao installed')

Installing torchao to resolve dependency error
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 36.0 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
torchao installed


### Step 3 — mount Drive for crash-safe checkpoints

In [4]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUTDIR = "/content/drive/MyDrive/codegen_regen"
os.makedirs(OUTDIR, exist_ok=True)
print("checkpoints will be saved to:", OUTDIR)

Mounted at /content/drive
checkpoints will be saved to: /content/drive/MyDrive/codegen_regen


### Step 4 — load the 164 HumanEval tasks

In [5]:
import json
TASKS = {t["task_id"]: t for t in json.load(open("data/human_eval.json"))}
print(len(TASKS), "tasks loaded")

164 tasks loaded


### Step 5 — load both models (CodeGen-350M + your LoRA adapter, and Qwen2.5-Coder-1.5B)

In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

device = "cuda"
dtype  = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16  # T4 -> fp16
print("compute dtype:", dtype)

# --- CodeGen-350M base + your LoRA adapter (base-completion model) ---
cg_tok  = AutoTokenizer.from_pretrained("Salesforce/codegen-350M-mono")
cg_base = AutoModelForCausalLM.from_pretrained("Salesforce/codegen-350M-mono", torch_dtype=dtype).to(device)
codegen = PeftModel.from_pretrained(cg_base, "finetuned-adapter").to(device).eval()

# --- Qwen2.5-Coder-1.5B-Instruct (instruction-tuned baseline) ---
qw_tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-1.5B-Instruct")
qwen   = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-Coder-1.5B-Instruct", torch_dtype=dtype).to(device).eval()
print("both models loaded")

compute dtype: torch.bfloat16


Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

[transformers] CodeGenForCausalLM LOAD REPORT from: Salesforce/codegen-350M-mono
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...19}.attn.causal_mask | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

both models loaded


### Step 6 — generation functions (correct protocol per model, 5 samples each)

In [9]:
N_SAMPLES, TEMP, TOP_P, MAX_NEW = 5, 0.6, 0.95, 192
GEN = dict(do_sample=True, temperature=TEMP, top_p=TOP_P,
           max_new_tokens=MAX_NEW, num_return_sequences=N_SAMPLES)

@torch.no_grad()
def gen_codegen(prompt):                       # base model: raw prompt continuation
    ids = cg_tok(prompt, return_tensors="pt").to(device)
    out = codegen.generate(**ids, pad_token_id=cg_tok.eos_token_id, **GEN)
    new = out[:, ids["input_ids"].shape[1]:]
    return cg_tok.batch_decode(new, skip_special_tokens=True)

@torch.no_grad()
def gen_qwen(prompt):                          # instruct model: chat template
    messages = [
        {"role":"system","content":"You are an expert Python programmer. Complete the function. Output only Python code."},
        {"role":"user","content":f"Complete this Python function:\n\n{prompt}"},
    ]
    text = qw_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids  = qw_tok(text, return_tensors="pt").to(device)
    out  = qwen.generate(**ids, pad_token_id=qw_tok.eos_token_id, **GEN)
    new  = out[:, ids["input_ids"].shape[1]:]
    return qw_tok.batch_decode(new, skip_special_tokens=True)

### Step 7 — generate all completions  ⏳ (the long cell — ~1.5–3 h)
Checkpoints to Drive every 5 tasks. **If Colab disconnects, just re-run this cell — it resumes.**

In [11]:
import json, os, time

def regenerate(name, fn):
    path = os.path.join(OUTDIR, f"completions_{name}.json")
    done = json.load(open(path)) if os.path.exists(path) else {}
    todo = [tid for tid in TASKS if tid not in done]
    print(f"[{name}] {len(done)} already done, {len(todo)} remaining")
    t0 = time.time()
    for i, tid in enumerate(todo):
        done[tid] = fn(TASKS[tid]["prompt"])
        if (i+1) % 5 == 0 or i == len(todo)-1:
            json.dump(done, open(path, "w"))
            rate = (time.time()-t0)/(i+1)
            print(f"  {len(done)}/{len(TASKS)}  ~{rate:.1f}s/task  (saved)")
    return done

cg_comps = regenerate("codegen", gen_codegen)
qw_comps = regenerate("qwen",    gen_qwen)
print("\n generation complete — both models have 5 completions for all 164 tasks")

[codegen] 0 already done, 164 remaining
  5/164  ~8.9s/task  (saved)
  10/164  ~8.5s/task  (saved)
  15/164  ~8.5s/task  (saved)
  20/164  ~8.7s/task  (saved)
  25/164  ~8.6s/task  (saved)
  30/164  ~8.5s/task  (saved)
  35/164  ~8.4s/task  (saved)
  40/164  ~8.4s/task  (saved)
  45/164  ~8.4s/task  (saved)
  50/164  ~8.3s/task  (saved)
  55/164  ~8.3s/task  (saved)
  60/164  ~8.3s/task  (saved)
  65/164  ~8.3s/task  (saved)
  70/164  ~8.3s/task  (saved)
  75/164  ~8.2s/task  (saved)
  80/164  ~8.2s/task  (saved)
  85/164  ~8.2s/task  (saved)
  90/164  ~8.2s/task  (saved)
  95/164  ~8.2s/task  (saved)
  100/164  ~8.2s/task  (saved)
  105/164  ~8.2s/task  (saved)
  110/164  ~8.2s/task  (saved)
  115/164  ~8.2s/task  (saved)
  120/164  ~8.2s/task  (saved)
  125/164  ~8.2s/task  (saved)
  130/164  ~8.2s/task  (saved)
  135/164  ~8.2s/task  (saved)
  140/164  ~8.2s/task  (saved)
  145/164  ~8.2s/task  (saved)
  150/164  ~8.2s/task  (saved)
  155/164  ~8.2s/task  (saved)
  160/164  ~8.2s/ta

### Step 8 — scoring utilities (naive vs. normalized — identical logic to reextract_experiment.py)

In [12]:
import re, subprocess, tempfile, textwrap
TIMEOUT = 5

def run_program(src):
    with tempfile.NamedTemporaryFile(suffix=".py", delete=False, mode="w") as f:
        f.write(src); tmp = f.name
    try:
        r = subprocess.run(["python", tmp], capture_output=True, text=True, timeout=TIMEOUT)
        if r.returncode == 0: return "OK"
        e = r.stderr[-300:]
        if "SyntaxError" in e or "IndentationError" in e: return "SYNTAX"
        if "AssertionError" in e: return "ASSERTION"
        return "RUNTIME"
    except subprocess.TimeoutExpired: return "TIMEOUT"
    except Exception: return "RUNTIME"

def import_header(prompt, ep):
    out = []
    for line in prompt.splitlines():
        if re.match(rf"\s*def\s+{re.escape(ep)}\b", line): break
        out.append(line)
    return "\n".join(out)

def strip_fences(c):
    if "```" in c:
        m = re.search(r"```(?:python)?\s*(.*?)```", c, re.DOTALL)
        if m: return m.group(1)
    return c

def normalize(completion, prompt, ep):
    c = strip_fences(completion)
    if re.search(rf"\bdef\s+{re.escape(ep)}\b", c):           # full function -> standalone
        lines = c.splitlines()
        for i, ln in enumerate(lines):
            if re.match(r"\s*(from |import |def )", ln):
                c = "\n".join(lines[i:]); break
        return import_header(prompt, ep) + "\n" + c
    return prompt + textwrap.indent(textwrap.dedent(c), "    ")   # bare body -> nest under def

def build(prompt, completion, test, ep, scheme):
    tail = "\n\n" + test + f"\n\ncheck({ep})\n"
    body = (prompt + completion) if scheme == "naive" else normalize(completion, prompt, ep)
    return body + tail

### Step 9 — score everything, compute pass@k, breakdown, and deltas

In [13]:
import numpy as np, pandas as pd
from collections import Counter

def pass_at_k(n, c, k):
    if n - c < k: return 1.0
    return 1.0 - np.prod(1.0 - k/np.arange(n-c+1, n+1))   # Chen et al. 2021

def evaluate(comps, scheme):
    per_task = []; cats = Counter()
    for tid, t in TASKS.items():
        cs = comps[tid]; n = len(cs); passes = 0
        for comp in cs:
            res = run_program(build(t["prompt"], comp, t["test"], t["entry_point"], scheme))
            cats[res] += 1
            if res == "OK": passes += 1
        per_task.append((n, passes))
    p1 = float(np.mean([pass_at_k(n, c, 1)          for n, c in per_task]))
    p5 = float(np.mean([pass_at_k(n, c, min(5, n))  for n, c in per_task]))
    return p1, p5, cats

rows, breakdown = [], []
for name, comps in [("codegen", cg_comps), ("qwen", qw_comps)]:
    for scheme in ["naive", "normalized"]:
        p1, p5, cats = evaluate(comps, scheme)
        rows.append({"model":name,"scheme":scheme,"pass@1":round(p1*100,1),"pass@5":round(p5*100,1)})
        breakdown.append({"model":name,"scheme":scheme,
                          **{k:cats.get(k,0) for k in ["OK","SYNTAX","ASSERTION","RUNTIME","TIMEOUT"]}})
        print(f"{name:8} {scheme:10}  pass@1={p1*100:5.1f}%  pass@5={p5*100:5.1f}%   {dict(cats)}")

res = pd.DataFrame(rows); bd = pd.DataFrame(breakdown)
res.to_csv(os.path.join(OUTDIR, "passk_corrected.csv"), index=False)
bd.to_csv(os.path.join(OUTDIR, "breakdown_corrected.csv"), index=False)

print("\n=== HEADLINE DELTA (normalized − naive) ===")
for name in ["codegen", "qwen"]:
    a = res[(res.model==name) & (res.scheme=="naive")].iloc[0]
    b = res[(res.model==name) & (res.scheme=="normalized")].iloc[0]
    print(f"{name:8} pass@1 {a['pass@1']:5.1f}% -> {b['pass@1']:5.1f}%  "
          f"(+{b['pass@1']-a['pass@1']:.1f} pts)   pass@5 {a['pass@5']:5.1f}% -> {b['pass@5']:5.1f}%")

print("\nSaved: passk_corrected.csv, breakdown_corrected.csv ->", OUTDIR)
display(res); display(bd)

codegen  naive       pass@1=  2.4%  pass@5=  8.5%   {'SYNTAX': 520, 'ASSERTION': 200, 'RUNTIME': 80, 'OK': 20}
codegen  normalized  pass@1=  0.5%  pass@5=  2.4%   {'SYNTAX': 731, 'RUNTIME': 17, 'ASSERTION': 68, 'OK': 4}
qwen     naive       pass@1=  0.0%  pass@5=  0.0%   {'SYNTAX': 820}
qwen     normalized  pass@1= 59.4%  pass@5= 82.3%   {'RUNTIME': 50, 'OK': 487, 'ASSERTION': 227, 'SYNTAX': 55, 'TIMEOUT': 1}

=== HEADLINE DELTA (normalized − naive) ===
codegen  pass@1   2.4% ->   0.5%  (+-1.9 pts)   pass@5   8.5% ->   2.4%
qwen     pass@1   0.0% ->  59.4%  (+59.4 pts)   pass@5   0.0% ->  82.3%

Saved: passk_corrected.csv, breakdown_corrected.csv -> /content/drive/MyDrive/codegen_regen


,model,scheme,pass@1,pass@5
0,codegen,naive,2.4,8.5
1,codegen,normalized,0.5,2.4
2,qwen,naive,0.0,0.0
3,qwen,normalized,59.4,82.3


,model,scheme,OK,SYNTAX,ASSERTION,RUNTIME,TIMEOUT
0,codegen,naive,20,520,200,80,0
1,codegen,normalized,4,731,68,17,0
2,qwen,naive,0,820,0,0,0
3,qwen,normalized,487,55,227,50,1
